# ELVES-Dwarf tutorial: quenched fraction

This notebook uses the public **ELVES-Dwarf v1 satellite candidate catalog** to calculate the quenched fraction of confirmed satellites. It is a compact, reproducible version of the original analysis notebook.

You will:

1. load the public FITS catalog;
2. define a bright, confirmed-satellite sample;
3. classify satellites with a color--magnitude cut; and
4. calculate binned quenched fractions with 68% Jeffreys binomial intervals.

**Requirements:** Python 3, NumPy, Astropy, and Matplotlib. In a new environment, install them with `pip install numpy astropy matplotlib`.

## 1. Load the catalog

The code first looks for the catalog in a local checkout of the website. If it is not present, Astropy reads the same file from the public data URL. This makes the notebook work both inside the repository and as a standalone download.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.stats import binom_conf_interval
from astropy.table import Table

CATALOG_NAME = "ELVES-Dwarf_master_cat_v1.fits"
CATALOG_URL = (
    "https://elves-surveys.github.io/data/elves-dwarf/" + CATALOG_NAME
)
LOCAL_CANDIDATES = (
    Path("../../data/elves-dwarf") / CATALOG_NAME,
    Path("public/data/elves-dwarf") / CATALOG_NAME,
)
catalog_source = next(
    (path for path in LOCAL_CANDIDATES if path.exists()),
    CATALOG_URL,
)

catalog = Table.read(catalog_source)
for column in catalog.colnames:
    if catalog[column].dtype.kind == "S":
        catalog[column] = catalog[column].astype(str)
print(f"Loaded {len(catalog)} rows from {catalog_source}")
print("Membership classes:")
for status in np.unique(catalog["status"]):
    print(f"  {status}: {np.count_nonzero(catalog['status'] == status)}")

## 2. Define the analysis sample

For this tutorial we use:

- objects whose catalog `status` is `Confirmed`;
- an absolute-magnitude limit of $M_V < -9$; and
- objects with a usable $g-i$ color, either measured directly or converted from a valid $g-r$ color.

We calculate absolute magnitude from the apparent Sersic magnitude and the host distance:

$$M_V = m_V - 5\log_{10}(D_{\rm host}/{\rm Mpc}) - 25.$$

When `gi_sersic` is missing but `gr_sersic` is valid, we use the relation from the original notebook, $(g-i)=1.53(g-r)-0.032$. Catalog placeholders such as `999` are treated as missing, not as physical colors.

In [ ]:
M_V_LIMIT = -9.0

confirmed = catalog[np.asarray(catalog["status"]).astype(str) == "Confirmed"].copy()
confirmed["M_V"] = confirmed["m_V_sersic"] - (
    5 * np.log10(confirmed["host_dist"]) + 25
)
bright = confirmed[confirmed["M_V"] < M_V_LIMIT].copy()

gi = np.asarray(np.ma.filled(bright["gi_sersic"], np.nan), dtype=float)
gr = np.asarray(np.ma.filled(bright["gr_sersic"], np.nan), dtype=float)
valid_gi = np.isfinite(gi) & (np.abs(gi) < 5)
valid_gr = np.isfinite(gr) & (np.abs(gr) < 5)
converted = ~valid_gi & valid_gr
gi[converted] = 1.53 * gr[converted] - 0.032

valid_color = np.isfinite(gi) & (np.abs(gi) < 5)
sample = bright[valid_color].copy()
sample["g_i_used"] = gi[valid_color]
sample["color_source"] = np.where(
    valid_gi[valid_color], "catalog g-i", "converted from g-r"
)

print(f"Confirmed satellites: {len(confirmed)}")
print(f"After M_V < {M_V_LIMIT:g}: {len(bright)}")
print(f"With a usable color: {len(sample)}")
print(f"Excluded for missing color: {len(bright) - len(sample)}")
sample[["name", "host", "M_V", "g_i_used", "color_source"]][:10]

## 3. Classify quenched satellites

Following the color--magnitude definition used in the original notebook, a satellite is classified as quenched when

$$(g-i) > -0.067M_V - 0.23.$$

This is a photometric classification. It is not a direct measurement of star-formation rate, and objects without a usable color are excluded from the denominator.

In [ ]:
sample["quenching_boundary"] = -0.067 * sample["M_V"] - 0.23
sample["quenched"] = sample["g_i_used"] > sample["quenching_boundary"]

n_total = len(sample)
n_quenched = int(np.count_nonzero(sample["quenched"]))
fraction = n_quenched / n_total
interval = binom_conf_interval(
    n_quenched,
    n_total,
    confidence_level=0.68,
    interval="jeffreys",
)

print(f"Quenched: {n_quenched}/{n_total}")
print(
    f"Overall quenched fraction: {fraction:.3f} "
    f"(68% Jeffreys interval: {interval[0]:.3f}--{interval[1]:.3f})"
)
sample[["name", "host", "M_V", "g_i_used", "quenched"]]

In [ ]:
is_quenched = np.asarray(sample["quenched"], dtype=bool)
magnitude_grid = np.linspace(-17.5, -8.5, 200)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    sample["M_V"][~is_quenched],
    sample["g_i_used"][~is_quenched],
    label="Star-forming side of cut",
    color="#168577",
    edgecolor="white",
    linewidth=0.7,
    s=55,
)
ax.scatter(
    sample["M_V"][is_quenched],
    sample["g_i_used"][is_quenched],
    label="Quenched side of cut",
    color="#d95f59",
    edgecolor="white",
    linewidth=0.7,
    s=55,
)
ax.plot(
    magnitude_grid,
    -0.067 * magnitude_grid - 0.23,
    color="#31363f",
    linestyle="--",
    label="Quenching boundary",
)
ax.set(xlabel=r"$M_V$", ylabel=r"$(g-i)$ used for classification")
ax.invert_xaxis()
ax.legend(frameon=False)
ax.grid(alpha=0.18)
plt.show()

## 4. Measure the quenched fraction versus stellar mass

For each stellar-mass bin, the estimator is simply $f_q=k/n$, where $k$ is the number classified as quenched and $n$ is the number with a usable color. We use a 68% Jeffreys binomial interval because the bins contain small samples and may have $k=0$ or $k=n$.

The bin edges below follow the compact calculation in the original notebook. Try changing them to see how small-number statistics affect the result.

In [ ]:
def binned_quenched_fraction(log_mass, quenched, bin_edges):
    """Return f_q and 68% Jeffreys intervals in stellar-mass bins."""
    rows = []
    for left, right in zip(bin_edges[:-1], bin_edges[1:]):
        in_bin = (log_mass >= left) & (log_mass < right)
        n = int(np.count_nonzero(in_bin))
        if n == 0:
            continue
        k = int(np.count_nonzero(quenched & in_bin))
        lower, upper = binom_conf_interval(
            k, n, confidence_level=0.68, interval="jeffreys"
        )
        rows.append(
            (left, right, np.mean(log_mass[in_bin]), n, k, k / n, lower, upper)
        )

    return Table(
        rows=rows,
        names=(
            "logM_left",
            "logM_right",
            "mean_logM",
            "N",
            "N_quenched",
            "f_quenched",
            "f_lower",
            "f_upper",
        ),
    )

In [ ]:
mass_bins = np.array([5.0, 6.0, 6.6, 7.5, 9.1])
binned = binned_quenched_fraction(
    np.asarray(sample["log_m_star"], dtype=float),
    np.asarray(sample["quenched"], dtype=bool),
    mass_bins,
)
for column in ("mean_logM", "f_quenched", "f_lower", "f_upper"):
    binned[column].info.format = ".3f"
binned

In [ ]:
x = np.asarray(binned["mean_logM"], dtype=float)
y = np.asarray(binned["f_quenched"], dtype=float)
lower = np.asarray(binned["f_lower"], dtype=float)
upper = np.asarray(binned["f_upper"], dtype=float)

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(
    x,
    y,
    yerr=np.vstack((y - lower, upper - y)),
    fmt="s",
    color="#168577",
    markeredgecolor="#075f52",
    markersize=8,
    linewidth=1.8,
)
for x_value, y_value, n in zip(x, y, binned["N"]):
    ax.annotate(f"N={n}", (x_value, y_value), xytext=(0, 11),
                textcoords="offset points", ha="center", fontsize=9)
ax.set(
    xlabel=r"$\log_{10}(M_\star/M_\odot)$",
    ylabel="Quenched fraction",
    xlim=(4.9, 9.2),
    ylim=(-0.05, 1.05),
)
ax.grid(alpha=0.18)
plt.show()

## Interpretation and next steps

This tutorial reports the fraction for the explicitly selected color-valid sample. It does **not** perform completeness weighting, statistical background subtraction, or a sensitivity analysis for alternative quenching definitions. Those choices should be revisited for a publication-level measurement.

Useful experiments:

- change `M_V_LIMIT` and the stellar-mass bins;
- compare direct $g-i$ measurements with colors converted from $g-r$;
- inspect how the result changes if objects without a usable color are assigned a classification from independent star-formation indicators;
- split the sample by host stellar mass or isolation; and
- join the satellite and host catalogs for additional host properties.

When publishing results from these data, cite the release paper listed in the [ELVES-Dwarf data documentation](https://elves-surveys.github.io/catalogs/elves-dwarf/).